# Chicago Closure Radar — Step 1: Data Collection

This notebook pulls all three real data sources:
1. **City of Chicago Business Licenses** (free, no key needed)
2. **City of Chicago Food Inspections** (free, no key needed)
3. **Yelp Open Dataset** (manual download required — see instructions below)

### Yelp Dataset Download
1. Go to https://business.yelp.com/data/resources/open-dataset/
2. Register with your academic/personal email
3. Download `yelp_dataset.tar` and unzip into `data/raw/yelp/`

**Or** use Kaggle (faster):
```bash
pip install kaggle
# Put kaggle.json in ~/.kaggle/
kaggle datasets download adamamer2001/yelp-complete-open-dataset-2024 -p data/raw/yelp/ --unzip
```

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from pathlib import Path
import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s')

RAW = Path('../data/raw')
GT  = Path('../data/ground_truth')
RAW.mkdir(parents=True, exist_ok=True)
GT.mkdir(parents=True, exist_ok=True)

## 1. Chicago Business Licenses

In [ ]:
from src.data.chicago_portal import fetch_business_licenses

licenses = fetch_business_licenses(RAW, refresh=False)
print(f'Total license records: {len(licenses):,}')
print(f'Closed businesses: {licenses["is_closed"].sum():,}')
licenses.head(3)

## 2. Chicago Food Inspections

In [ ]:
from src.data.chicago_portal import fetch_food_inspections

inspections = fetch_food_inspections(RAW, refresh=False)
print(f'Total inspection records: {len(inspections):,}')
print(f'"Out of Business" inspections: {inspections["is_out_of_business"].sum():,}')
inspections['results'].value_counts().head(10)

## 3. Build Ground Truth (Closure Labels)

In [ ]:
from src.data.chicago_portal import build_ground_truth

gt = build_ground_truth(licenses, inspections)
gt.to_parquet(GT / 'ground_truth.parquet', index=False)

print(gt['is_closed'].value_counts())
gt[gt['is_closed']].sort_values('closure_date', ascending=False).head(10)

## 4. Yelp Open Dataset — Chicago Filter

In [ ]:
from src.data.yelp_loader import load_businesses, load_reviews

YELP = RAW / 'yelp'
businesses = load_businesses(YELP)
businesses.to_parquet('../data/processed/yelp_chicago_businesses.parquet', index=False)
print(f'Chicago businesses: {len(businesses):,}')
businesses[['name', 'stars', 'review_count', 'categories', 'is_open']].head(10)

In [ ]:
reviews = load_reviews(YELP, set(businesses['business_id']))
reviews.to_parquet('../data/processed/yelp_chicago_reviews.parquet', index=False)
print(f'Total reviews: {len(reviews):,}')
print(f'Date range: {reviews["date"].min().date()} → {reviews["date"].max().date()}')
reviews.head(3)

## Quick Sanity Check: Known Closures

Spot-check against known closed Chicago spots:

In [ ]:
known_closed = ['Book Cellar', 'Metropolis Coffee', 'Intelligentsia', 'Argo Tea']
mask = businesses['name'].str.contains('|'.join(known_closed), case=False, na=False)
businesses[mask][['name', 'stars', 'review_count', 'is_open', 'categories']]